In [14]:
import pandas as pd

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Función simple: clasificar edad en 'adulto' o 'menor'
def clasificar_edad(edad):
    if pd.isnull(edad):
        return 'desconocido'
    elif edad < 18:
        return 'menor'
    else:
        return 'adulto'

df['grupo_edad'] = df['Age'].apply(clasificar_edad)

# Verificar resultado
print(df[['Name', 'Age', 'grupo_edad']].head(10))

                                                Name   Age   grupo_edad
0                            Braund, Mr. Owen Harris  22.0       adulto
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0       adulto
2                             Heikkinen, Miss. Laina  26.0       adulto
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0       adulto
4                           Allen, Mr. William Henry  35.0       adulto
5                                   Moran, Mr. James   NaN  desconocido
6                            McCarthy, Mr. Timothy J  54.0       adulto
7                     Palsson, Master. Gosta Leonard   2.0        menor
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  27.0       adulto
9                Nasser, Mrs. Nicholas (Adele Achem)  14.0        menor


In [2]:
# Ejemplo 1: columna calculada directamente
df['tarifa_por_familiar'] = df['Fare'] / (df['SibSp'] + df['Parch'] + 1)
# El +1 incluye al propio pasajero para evitar división por cero

# Ejemplo 2: columna binaria desde una condición
df['viajaba_solo'] = (df['SibSp'] + df['Parch'] == 0).astype(int)
# 1 si viajaba solo, 0 si tenía familiares a bordo

print(df[['Name', 'SibSp', 'Parch', 'viajaba_solo', 'tarifa_por_familiar']].head(8))

                                                Name  SibSp  Parch  \
0                            Braund, Mr. Owen Harris      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...      1      0   
2                             Heikkinen, Miss. Laina      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)      1      0   
4                           Allen, Mr. William Henry      0      0   
5                                   Moran, Mr. James      0      0   
6                            McCarthy, Mr. Timothy J      0      0   
7                     Palsson, Master. Gosta Leonard      3      1   

   viajaba_solo  tarifa_por_familiar  
0             0              3.62500  
1             0             35.64165  
2             1              7.92500  
3             0             26.55000  
4             1              8.05000  
5             1              8.45830  
6             1             51.86250  
7             0              4.21500  


In [3]:
# Recodificar la columna 'Sex'
df['sexo_num'] = df['Sex'].map({'male': 0, 'female': 1})

# Recodificar 'Embarked' (puerto de embarque)
df['puerto'] = df['Embarked'].map({'C': 'Cherburgo', 'Q': 'Queenstown', 'S': 'Southampton'})

print(df[['Sex', 'sexo_num', 'Embarked', 'puerto']].head(8))

      Sex  sexo_num Embarked       puerto
0    male         0        S  Southampton
1  female         1        C    Cherburgo
2  female         1        S  Southampton
3  female         1        S  Southampton
4    male         0        S  Southampton
5    male         0        Q   Queenstown
6    male         0        S  Southampton
7    male         0        S  Southampton


In [4]:
# ── 1. Tamaño del grupo familiar ──────────────────────────────
df['tam_familia'] = df['SibSp'] + df['Parch'] + 1

# ── 2. ¿Viajaba solo? ─────────────────────────────────────────
df['viajaba_solo'] = (df['tam_familia'] == 1).astype(int)

# ── 3. Título del pasajero (extraído del nombre) ──────────────
# Los nombres tienen formato: "Apellido, Título. Nombre"
df['titulo'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

print("Títulos únicos encontrados:")
print(df['titulo'].value_counts())

# ── 4. Categoría de tarifa ────────────────────────────────────
df['categoria_tarifa'] = pd.cut(
    df['Fare'],
    bins=[0, 10, 30, 100, 600],
    labels=['muy_baja', 'baja', 'media', 'alta']
)

# Vista final de las nuevas columnas
cols_nuevas = ['Name', 'tam_familia', 'viajaba_solo', 'titulo', 'Fare', 'categoria_tarifa']
print(df[cols_nuevas].head(10))

Títulos únicos encontrados:
titulo
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
                                                Name  tam_familia  \
0                            Braund, Mr. Owen Harris            2   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...            2   
2                             Heikkinen, Miss. Laina            1   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)            2   
4                           Allen, Mr. William Henry            1   
5                                   Moran, Mr. James            1   
6                            McCarthy, Mr. Timothy J            1   
7                     Palsson, Master. Gosta Le

In [16]:
# Estas dos formas son equivalentes:

# Con lambda
df['titulo'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

# Con función definida
def extraer_titulo(nombre):
    return nombre.split(',')[1].split('.')[0].strip()

df['titulo'] = df['Name'].apply(extraer_titulo)

In [33]:
# Ejercicio 1 — apply() con lógica propia
#  Crea una columna llamada nivel_tarifa usando apply() con la siguiente lógica:
# Fare < 10 → "económico"
# Fare entre 10 y 50 → "estándar"
# Fare > 50 → "premium"
# Si el valor es nulo → "desconocido"

def categoria_tarifa(fare):
  if pd.isnull(fare):
    return "tarifa no guardada"
  elif fare < 10:
    return "tarifa economica"
  elif 10 <= fare <=50:
    return "tarifa estandar"
  elif fare > 50:
    return "tarifa premium"
  else:
    return "tarifa desconocida"

df['nivel_tarifa'] = df['Fare'].apply(categoria_tarifa)

print(df[['Fare', 'Name', 'nivel_tarifa']].head(10))

# Ejercicio 2 — Columna derivada
#  El dataset tiene SibSp (hermanos/cónyuge) y Parch (padres/hijos). Crea:
# Una columna tam_familia con el tamaño total del grupo (incluyéndose)
# Una columna tipo_viajero usando apply() con esta lógica:
# Tamaño 1 → "solo"
# Tamaño 2 o 3 → "grupo_pequeño"
# Tamaño 4 o más → "grupo_grande"

df['tam_familiar'] = df['SibSp'] + df['Parch'] + 1

def tipo_viajero(tam_familiar):
  if tam_familiar == 1:
    return "solo"
  elif tam_familiar <=3:
    return "grupo pequeño"
  else:
    return "grupo grande"
df['tipo_viajero'] = df['tam_familiar'].apply(tipo_viajero)
print(df[['Name', 'SibSp', 'Parch', 'tam_familiar', 'tipo_viajero']].head(10))

# Ejercicio 3 — map() para recodificar
# La columna Pclass tiene valores 1, 2, 3. Usa map() para crear una columna clase_nombre con los valores:
# 1 → "Primera"
# 2 → "Segunda"
# 3 → "Tercera"
# Luego muestra cuántos pasajeros hay en cada clase usando value_counts().

df['clase_nombre'] = df['Pclass'].map({1: 'Primera', 2: 'Segunda', 3: 'Tercera'})
print(df['clase_nombre'].value_counts())

# Ejercicio 4 — Análisis con variables nuevas
#  Usando las columnas que creaste hoy, responde con código:
# ¿Cuál es la tasa de supervivencia de quienes viajaban solos vs. acompañados?

supervivencia_solo = df[df['tipo_viajero'] == 'solo']['Survived'].mean()
supervivencia_acompañados = df[df['tipo_viajero'] != 'solo']['Survived'].mean()
print(f"Tasa de supervivencia de quienes viajaban solo: {supervivencia_solo.round(2)}")
print(f"Tasa de supervivencia de quienes viajaban acompañados: {supervivencia_acompañados.round(2)}")

respuesta_1 = (
    "La tasa de supervivencia de quienes viajaban solos era de un 30%"
    " y de quienes viajaban acompñados era de un 51%."
    " Esta diferencia se da en base a que la mayoria de quienes viajaban solos eran hombres"
    " y los acompañados sobrevientes eran madres junto a sus hijos."
    )
print (respuesta_1)
# ¿Qué título tiene la mayor tasa de supervivencia?

supervivencia_titulo = df.groupby('titulo')['Survived'].mean()
print(f"Tasa de supervivencia por título: {supervivencia_titulo.round(2)}")

respuesta_2= (
    " La tasa de supervivencia por titulo nos muestra que Mrs tienen un 79% contra un "
    "16 % de Mr, igual se aprecia que titulos de mayor jerarquia tienen mayor tasa de supervivencia."
    " El titulo capitan apreciamos que tiene un 0% de supervivencia dado que se hundió con su barco."
)
print (respuesta_2)

# ¿Hay diferencia en la tarifa promedio entre quienes sobrevivieron y quienes no?

promedio_tarifa = df.groupby('Survived')['Fare'].mean()
print(f"Promedio de tarifa por supervivencia: {promedio_tarifa.round(2)}")

respuesta_3 = (
    " La tarifa mas baja tuvo un promedio de supervivencia de un 22.12 % "
    " mientras que la la tarifa mas alta tuvo un promedio de supervivencia de 48.40%. "
    " Esto nos demuestra que las clases mas altas tuvieron mayer acceso a botes "
    " y posibilidades de salvarse que las clases bajas del barco"
)
print (respuesta_3)

# Para cada pregunta, muestra el resultado como un groupby y escribe una oración de conclusión con el número concreto.



      Fare                                               Name  \
0   7.2500                            Braund, Mr. Owen Harris   
1  71.2833  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2   7.9250                             Heikkinen, Miss. Laina   
3  53.1000       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4   8.0500                           Allen, Mr. William Henry   
5   8.4583                                   Moran, Mr. James   
6  51.8625                            McCarthy, Mr. Timothy J   
7  21.0750                     Palsson, Master. Gosta Leonard   
8  11.1333  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)   
9  30.0708                Nasser, Mrs. Nicholas (Adele Achem)   

       nivel_tarifa  
0  tarifa economica  
1    tarifa premium  
2  tarifa economica  
3    tarifa premium  
4  tarifa economica  
5  tarifa economica  
6    tarifa premium  
7   tarifa estandar  
8   tarifa estandar  
9   tarifa estandar  
                                          

In [34]:
# Pregunta 1 — conclusión correcta
supervivencia_solo = df[df['tipo_viajero'] == 'solo']['Survived'].mean()
supervivencia_acomp = df[df['tipo_viajero'] != 'solo']['Survived'].mean()

print(f"Solos: {supervivencia_solo:.0%} | Acompañados: {supervivencia_acomp:.0%}")
print("Los pasajeros que viajaban solos sobrevivieron a una tasa del "
      f"{supervivencia_solo:.0%}, frente al {supervivencia_acomp:.0%} de quienes "
      "iban acompañados. Se requiere análisis adicional para explicar la causa.")

# Pregunta 2 — responder la pregunta con idxmax()
surv_titulo = df.groupby('titulo')['Survived'].mean()
titulo_max = surv_titulo.idxmax()
tasa_max = surv_titulo.max()

print(f"\nTasa por título:\n{surv_titulo.round(2)}")
print(f"\nEl título con mayor supervivencia es '{titulo_max}' con {tasa_max:.0%}.")

# Pregunta 3 — conclusión correcta
tarifa_promedio = df.groupby('Survived')['Fare'].mean()
print(f"\nTarifa promedio — No sobrevivieron: ${tarifa_promedio[0]:.2f} | "
      f"Sobrevivieron: ${tarifa_promedio[1]:.2f}")
print("Los sobrevivientes pagaron en promedio $48.40, más del doble que quienes "
      "no sobrevivieron ($22.12), lo que sugiere correlación entre clase económica "
      "y supervivencia.")

Solos: 30% | Acompañados: 51%
Los pasajeros que viajaban solos sobrevivieron a una tasa del 30%, frente al 51% de quienes iban acompañados. Se requiere análisis adicional para explicar la causa.

Tasa por título:
titulo
Capt            0.00
Col             0.50
Don             0.00
Dr              0.43
Jonkheer        0.00
Lady            1.00
Major           0.50
Master          0.57
Miss            0.70
Mlle            1.00
Mme             1.00
Mr              0.16
Mrs             0.79
Ms              1.00
Rev             0.00
Sir             1.00
the Countess    1.00
Name: Survived, dtype: float64

El título con mayor supervivencia es 'Lady' con 100%.

Tarifa promedio — No sobrevivieron: $22.12 | Sobrevivieron: $48.40
Los sobrevivientes pagaron en promedio $48.40, más del doble que quienes no sobrevivieron ($22.12), lo que sugiere correlación entre clase económica y supervivencia.
